# When MAS Beats a Single LLM Call

A single `chat.completions.create` call is the **simplest** way to use an LLM — and often the **best**. You send a prompt, get an answer, and move on. No orchestration, no state machine, no agent registry.

A **multi-agent system (MAS)** earns its complexity only when a task genuinely benefits from:

- **Decomposition** — breaking a hard problem into smaller sub-goals.
- **Specialization** — different system prompts for different expertise.
- **Parallel research** — independent sub-queries run at the same time.
- **Critique loops** — a generator and critic iterate until quality gates pass.
- **Failure isolation** — retry one failed step without discarding good work elsewhere.

This notebook maps **when decomposition wins**, shows **counterexamples** where one call is superior, and builds a **cost and latency analysis** you can reuse in design reviews. Everything is self-contained with runnable Python — no prior notebooks or external services required.

---

## Scenario: Production Release Readiness Review

A team is about to deploy the **Payments API** to production. They need to verify authentication docs, migration steps, test coverage, and a deploy runbook — then produce a sign-off checklist. Some steps can run in parallel; a critic must approve the final runbook before release.

We compare how a **single LLM call**, a **decomposed pipeline**, and a **multi-agent crew** handle the same goal — with measurable cost, latency, and quality trade-offs.

In [ ]:
"""
Shared environment: internal docs corpus and release context.
"""

import asyncio
import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

# Internal documentation chunks (semantic memory for all patterns)
RELEASE_DOCS: list[dict[str, str]] = [
    {
        "id": "auth-01",
        "topic": "authentication",
        "text": "All Payments API requests require JWT bearer token in Authorization header. Tokens expire after 8 hours.",
    },
    {
        "id": "migrate-02",
        "topic": "migrations",
        "text": "Run alembic upgrade head on staging before production. Never skip migration dry-run in CI.",
    },
    {
        "id": "test-03",
        "topic": "testing",
        "text": "Payments integration tests use httpx.AsyncClient. Database fixtures live in conftest.py with session scope.",
    },
    {
        "id": "deploy-04",
        "topic": "deployment",
        "text": "Deploy via ArgoCD: staging smoke test must pass before prod promotion. Roll back on failed health checks.",
    },
    {
        "id": "oncall-05",
        "topic": "oncall",
        "text": "SEV-1 payments incidents page Team Ledger. Post updates every 15 minutes in #incidents-payments.",
    },
]

RELEASE_GOAL = (
    "Prepare a production release checklist for Payments API covering "
    "auth, migrations, tests, deployment, and on-call escalation."
)


def search_docs(keyword: str) -> list[dict[str, str]]:
    """Keyword search over release documentation."""
    kw = keyword.lower()
    return [
        d for d in RELEASE_DOCS
        if kw in d["topic"].lower() or kw in d["text"].lower()
    ]


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


print(f"Release docs loaded: {len(RELEASE_DOCS)}")
print(f"Goal: {RELEASE_GOAL}")

---

## 1. The Single-Call Baseline

The simplest pattern: one prompt → one model → one answer.

```
user question ──► ChatCompletion ──► final answer
```

| Strength | Weakness |
|----------|----------|
| Lowest latency and operational surface | Many constraints crammed into one context |
| Easiest to test and version | No natural place for per-subtask tool retries |
| Predictable cost (one billable call) | Single point of reasoning failure |

**Rule of thumb:** treat single-call as the **default**. Add agents only when you can name a concrete decomposition benefit.

In [ ]:
def single_call_release_checklist(goal: str) -> dict[str, Any]:
    """
    Offline stand-in for one-shot ChatCompletion.
    Simulates answering from model memory — no retrieval step.
    """
    start = time.perf_counter()

    # Single call tries to cover everything at once — may miss details
    answer = (
        "Release checklist:\n"
        "1. Verify JWT auth is configured.\n"
        "2. Run database migrations.\n"
        "3. Run integration tests.\n"
        "4. Deploy to production.\n"
        "(Note: no doc citations — model relied on training data.)"
    )

    elapsed_ms = (time.perf_counter() - start) * 1000
    return {
        "pattern": "single_call",
        "model_calls": 1,
        "tokens_in_est": 2800,
        "tokens_out_est": 600,
        "latency_ms": round(elapsed_ms + 1800, 1),  # simulated API latency
        "citations": [],
        "answer": answer,
    }


single = single_call_release_checklist(RELEASE_GOAL)
print(f"Pattern: {single['pattern']} | Calls: {single['model_calls']} | Latency: {single['latency_ms']} ms")
print(single["answer"])

---

## 2. Decomposition Benefits

Decomposition splits a task into **sub-goals** with clearer prompts and smaller context per step.

```
                    ┌──► retrieve auth docs ──► summarize JWT steps
release goal ───────┼──► retrieve migration docs ──► verify alembic
                    └──► synthesize checklist ──► final runbook
```

| Subtask | Why separate |
|---------|-------------|
| **Retrieval** | Ground answers in `RELEASE_DOCS`, not model memory |
| **Diagnosis** | Narrow system prompt → fewer hallucinated steps |
| **Synthesis** | Only sees **verified** sub-results |

Each box in the diagram can be a separate agent, a separate graph node, or a separate function — the benefit is **focused context**, not the label you put on it.

In [ ]:
def decomposed_release_review() -> dict[str, Any]:
    """Sequential decomposition: retrieve → diagnose → synthesize."""
    start = time.perf_counter()
    trace = []

    # Step 1: retrieve (grounded in docs)
    auth_hits = search_docs("authentication")
    migrate_hits = search_docs("migrations")
    test_hits = search_docs("testing")
    deploy_hits = search_docs("deployment")
    trace.append({"step": "retrieve", "chunks": [h["id"] for h in auth_hits + migrate_hits + test_hits + deploy_hits]})

    # Step 2: diagnose gaps
    diagnosis = "All four domains have documentation. Auth and deploy are critical path."
    trace.append({"step": "diagnose", "detail": diagnosis})

    # Step 3: synthesize with citations
    lines = []
    for hits in (auth_hits, migrate_hits, test_hits, deploy_hits):
        if hits:
            h = hits[0]
            lines.append(f"- [{h['id']}] {h['text']}")
    synthesis = "Release checklist (grounded):\n" + "\n".join(lines)
    trace.append({"step": "synthesize", "lines": len(lines)})

    elapsed_ms = (time.perf_counter() - start) * 1000
    return {
        "pattern": "decomposed_sequential",
        "model_calls": 3,
        "tokens_in_est": 3200,
        "tokens_out_est": 900,
        "latency_ms": round(elapsed_ms + 4200, 1),
        "citations": [h["id"] for h in auth_hits + migrate_hits + test_hits + deploy_hits],
        "answer": synthesis,
        "trace": trace,
    }


decomposed = decomposed_release_review()
print(f"Pattern: {decomposed['pattern']} | Citations: {decomposed['citations']}")
print(decomposed["answer"])

---

## 3. Specialization — Role-Specific System Prompts

MAS lets each participant carry a **tight system message** instead of one mega-prompt listing every rule.

| Agent | System focus | Release example |
|-------|--------------|-----------------|
| **Retriever** | Search policy, return chunk IDs only | Return `auth-01` for JWT questions |
| **Backend expert** | FastAPI + migration conventions | Explain alembic workflow |
| **QA engineer** | pytest + httpx patterns | Draft test verification steps |
| **Tech writer** | User-facing tone, numbered checklist | Final release runbook |

When prompts would conflict if combined — e.g. "be terse" vs "be exhaustive" — specialization is a strong signal to split agents.

In [ ]:
@dataclass
class SpecialistAgent:
    name: str
    system_prompt: str
    handler: Callable[[str], str]


SPECIALISTS = [
    SpecialistAgent(
        name="retriever",
        system_prompt="Return only relevant doc IDs. Never invent content.",
        handler=lambda q: ",".join(h["id"] for h in search_docs("auth")[:1]),
    ),
    SpecialistAgent(
        name="backend_expert",
        system_prompt="Explain migrations and API auth. Cite retrieved chunks.",
        handler=lambda _: search_docs("migrations")[0]["text"],
    ),
    SpecialistAgent(
        name="qa_engineer",
        system_prompt="Propose test verification steps. Never invent dependencies.",
        handler=lambda _: search_docs("testing")[0]["text"],
    ),
    SpecialistAgent(
        name="tech_writer",
        system_prompt="Merge specialist outputs into a numbered release checklist.",
        handler=lambda parts: "RELEASE CHECKLIST:\n" + "\n".join(f"{i+1}. {p}" for i, p in enumerate(parts)),
    ),
]


def specialized_pipeline(goal: str) -> dict[str, Any]:
    trace = []
    parts = []
    for agent in SPECIALISTS[:-1]:
        output = agent.handler(goal)
        trace.append({"agent": agent.name, "system": agent.system_prompt[:50] + "...", "output": output[:60]})
        parts.append(output)
    writer = SPECIALISTS[-1]
    final = writer.handler(parts)
    trace.append({"agent": writer.name, "output": final[:80]})
    return {
        "pattern": "specialist_sequential",
        "model_calls": len(SPECIALISTS),
        "tokens_in_est": 3800,
        "tokens_out_est": 1100,
        "latency_ms": 5100,
        "answer": final,
        "trace": trace,
    }


specialized = specialized_pipeline(RELEASE_GOAL)
print(specialized["answer"])
print("\nAgent trace:")
for step in specialized["trace"]:
    print(f"  [{step['agent']}]")

---

## 4. Parallel Research

Independent sub-queries can run **concurrently** — something a single sequential completion cannot do unless the model *simulates* parallelism in text.

```
         ┌─ agent A: search docs for authentication
task ────┼─ agent B: search docs for migrations
         └─ agent C: search docs for testing
                    │
                    ▼
              merge + dedupe ──► planner synthesizes
```

Parallel fan-out reduces **wall-clock latency** even when total token usage stays similar. This is one of the clearest cases where MAS beats a single call.

In [ ]:
async def research_agent(name: str, keyword: str) -> dict[str, Any]:
    """Simulate async doc search (I/O-bound)."""
    await asyncio.sleep(0.08)
    hits = search_docs(keyword)
    return {"agent": name, "keyword": keyword, "hits": hits}


async def parallel_research(keywords: list[str]) -> list[dict[str, Any]]:
    tasks = [research_agent(f"researcher_{kw}", kw) for kw in keywords]
    return await asyncio.gather(*tasks)


def merge_research_results(results: list[dict[str, Any]]) -> str:
    by_id: dict[str, str] = {}
    for r in results:
        for h in r["hits"]:
            by_id[h["id"]] = h["text"]
    return "\n".join(f"- [{k}] {v}" for k, v in sorted(by_id.items()))


async def parallel_release_review() -> dict[str, Any]:
    start = time.perf_counter()
    keywords = ["authentication", "migrations", "testing", "deployment", "oncall"]
    results = await parallel_research(keywords)
    merged = merge_research_results(results)
    elapsed_ms = (time.perf_counter() - start) * 1000

    return {
        "pattern": "parallel_research_merge",
        "model_calls": len(keywords) + 1,  # researchers + synthesizer
        "tokens_in_est": 4000,
        "tokens_out_est": 1200,
        "latency_ms": round(elapsed_ms + 2200, 1),  # parallel saves wall-clock
        "citations": list({h["id"] for r in results for h in r["hits"]}),
        "answer": f"Parallel research findings:\n{merged}",
        "branches": len(keywords),
    }


parallel = await parallel_release_review()
print(f"Pattern: {parallel['pattern']} | Branches: {parallel['branches']} | Latency: {parallel['latency_ms']} ms")
print(parallel["answer"])

---

## 5. Critique Loops — Generator and Critic

A **critic** agent reviews drafts until quality gates pass. This is the evaluator–optimizer pattern.

```
draft ──► critic ──► REVISE? ──► draft ──► critic ──► APPROVE ──► ship
```

| Benefit | Cost |
|---------|------|
| Catches missing citations, unsafe commands | 2× or more model calls per iteration |
| Enforces compliance checklists | Needs clear termination (`APPROVE` token or max rounds) |

Use critique loops when **quality matters more than speed** — deploy runbooks, legal summaries, security reviews.

In [ ]:
MAX_ROUNDS = 3
APPROVE_TOKEN = "APPROVE"


def draft_deploy_runbook() -> str:
    return (
        "Deploy runbook:\n"
        "1. Run alembic upgrade head.\n"
        "2. Deploy via ArgoCD.\n"
        "(missing: JWT verification step)"
    )


def critic_review(runbook: str) -> str:
    """Critic enforces auth check and doc citation before approval."""
    issues = []
    if "jwt" not in runbook.lower() and "bearer" not in runbook.lower():
        issues.append("add JWT bearer token verification before deploy")
    if "[" not in runbook:  # no doc citation
        issues.append("cite at least one doc chunk ID")
    if not issues:
        return APPROVE_TOKEN
    return "REVISE: " + "; ".join(issues)


def critique_loop() -> dict[str, Any]:
    draft = draft_deploy_runbook()
    trace = []
    rounds = 0

    for round_i in range(1, MAX_ROUNDS + 1):
        feedback = critic_review(draft)
        trace.append({"round": round_i, "critic": feedback})
        rounds = round_i
        if feedback.startswith(APPROVE_TOKEN):
            break
        if "jwt" in feedback.lower():
            auth_doc = search_docs("authentication")[0]
            draft += f"\n3. Verify [{auth_doc['id']}] {auth_doc['text']}"
        if "cite" in feedback.lower():
            deploy_doc = search_docs("deployment")[0]
            draft += f"\n4. Follow [{deploy_doc['id']}] {deploy_doc['text']}"

    return {
        "pattern": "critique_loop",
        "model_calls": rounds * 2,
        "tokens_in_est": 5200,
        "tokens_out_est": 1800,
        "latency_ms": 6800,
        "rounds": rounds,
        "approved": trace[-1]["critic"].startswith(APPROVE_TOKEN),
        "answer": draft,
        "trace": trace,
    }


critique = critique_loop()
print(f"Approved: {critique['approved']} after {critique['rounds']} round(s)")
for step in critique["trace"]:
    print(f"  Round {step['round']}: {step['critic']}")
print(f"\nFinal runbook:\n{critique['answer']}")

---

## 6. Failure Isolation

In MAS, a failed **tool call** or bad **retrieval** can be retried **inside one agent** without discarding good work from others.

```
planner (ok) ──► retriever (fail) ──► retry retriever only
              └──► writer (ok) ──► preserved in shared state
```

With a single call, failure often means **regenerate everything** — including reasoning that was already correct.

In [ ]:
class AgentStepError(Exception):
    pass


def run_agent_step(name: str, fn: Callable[[dict], Any], state: dict, retries: int = 2) -> None:
    for attempt in range(1, retries + 2):
        try:
            state[name] = fn(state)
            print(f"  {name}: success on attempt {attempt}")
            return
        except AgentStepError as exc:
            print(f"  {name}: attempt {attempt} failed — {exc}")
    state[name] = None
    print(f"  {name}: gave up after {retries + 1} attempts")


def flaky_doc_search(_state: dict) -> str:
    flaky_doc_search.calls += 1
    if flaky_doc_search.calls < 2:
        raise AgentStepError("doc search timeout")
    return search_docs("authentication")[0]["text"]


flaky_doc_search.calls = 0

shared_state: dict[str, Any] = {"goal": "Verify auth before release"}
print("MAS with failure isolation:")
run_agent_step("planner", lambda s: "plan: check auth, migrations, tests", shared_state)
run_agent_step("retriever", flaky_doc_search, shared_state)
run_agent_step("writer", lambda s: f"Checklist item 1: {s.get('retriever')}", shared_state)
print(f"\nPlanner preserved: {shared_state.get('planner')}")
print(f"Final writer output: {shared_state.get('writer')}")

---

## 7. Counterexamples — When a Single Call Wins

MAS is not free. Prefer **one completion** when:

| Scenario | Why single call |
|----------|----------------|
| Short FAQ, fixed domain | Coordination overhead exceeds benefit |
| Strict latency SLA (<2s) | Serial agent turns add seconds |
| Deterministic transform | Code pipeline, not dialogue |
| Tiny context, no tools | Decomposition adds bugs |
| Regulated audit needs one prompt hash | Fewer moving parts to certify |

**Rule of thumb:** if you cannot draw **two boxes** with genuinely different responsibilities, stay with one call.

In [ ]:
SINGLE_CALL_BETTER = [
    {"task": "Translate deploy notice to Spanish", "reason": "No retrieval or tools needed"},
    {"task": "Classify incident severity (SEV-1 to SEV-4)", "reason": "Single label, low ambiguity"},
    {"task": "Rewrite error message for clarity", "reason": "Style-only edit"},
    {"task": "Summarize a 200-word status update", "reason": "Fits in one context window"},
]

MAS_BETTER = [
    {"task": "Research auth + migrations + tests in parallel for release", "reason": "Parallel fan-out"},
    {"task": "Generate + review deploy runbook for compliance", "reason": "Critique loop"},
    {"task": "Debug flaky 401 across docs, tests, and config", "reason": "Specialist decomposition"},
    {"task": "Build feature spanning API + tests + docs", "reason": "Role-specific prompts"},
]

print("Prefer SINGLE call:")
for row in SINGLE_CALL_BETTER:
    print(f"  • {row['task']}\n    → {row['reason']}")

print("\nPrefer MAS:")
for row in MAS_BETTER:
    print(f"  • {row['task']}\n    → {row['reason']}")

---

## 8. Cost and Latency Analysis

Rough **relative** costs for the Payments API release review (illustrative — tune to your model pricing).

| Pattern | Model calls | Tokens (in/out est.) | Latency (p95 est.) | Failure blast radius |
|---------|-------------|------------------------|--------------------|-----------------------|
| Single call | 1 | 2.8k / 0.6k | ~1.8s | Whole answer |
| Decomposed sequential | 3 | 3.2k / 0.9k | ~4.2s | Per step |
| Specialist pipeline | 4 | 3.8k / 1.1k | ~5.1s | Per agent |
| Parallel + merge | 6 | 4.0k / 1.2k | ~2.9s | Per branch |
| Critique loop (2 rounds) | 4 | 5.2k / 1.8k | ~6.8s | Loop bounded |

MAS can **reduce tokens per call** (smaller prompts) while **increasing call count**. Measure on **your** traces before committing.

In [ ]:
COST_ROWS = [
    {"pattern": "single_call", "calls": 1, "tokens_in": 2800, "tokens_out": 600, "usd_est": 0.005, "p95_sec": 1.8, "citations": 0},
    {"pattern": "decomposed_sequential", "calls": 3, "tokens_in": 3200, "tokens_out": 900, "usd_est": 0.008, "p95_sec": 4.2, "citations": 4},
    {"pattern": "specialist_sequential", "calls": 4, "tokens_in": 3800, "tokens_out": 1100, "usd_est": 0.010, "p95_sec": 5.1, "citations": 3},
    {"pattern": "parallel_research_merge", "calls": 6, "tokens_in": 4000, "tokens_out": 1200, "usd_est": 0.011, "p95_sec": 2.9, "citations": 5},
    {"pattern": "critique_loop", "calls": 4, "tokens_in": 5200, "tokens_out": 1800, "usd_est": 0.014, "p95_sec": 6.8, "citations": 2},
]

print(f"{'Pattern':<28} {'Calls':>5} {'In':>6} {'Out':>5} {'USD':>6} {'p95s':>5} {'Cites':>5}")
print("-" * 68)
for row in sorted(COST_ROWS, key=lambda r: r["usd_est"]):
    print(
        f"{row['pattern']:<28} {row['calls']:>5} {row['tokens_in']:>6} "
        f"{row['tokens_out']:>5} {row['usd_est']:>6.3f} {row['p95_sec']:>5.1f} {row['citations']:>5}"
    )

print("\nCheapest: single_call")
print("Best citation coverage: parallel_research_merge")
print("Highest quality ceiling: critique_loop (when critic catches real gaps)")

---

## 9. Decision Framework

Use this flow when choosing architecture:

```
Start: Can one prompt + optional tools answer reliably?
  yes ──► single call (stop)
  no ──► need parallel evidence? ──► parallel MAS
       need quality gate? ──► critic loop
       need distinct expertise? ──► specialist agents
       need resilient tools? ──► isolated agent per tool domain
```

In [ ]:
def recommend_architecture(task: str) -> str:
    t = task.lower()
    if any(k in t for k in ("translate", "classify", "rewrite", "summarize")):
        return "single_call"
    if any(k in t for k in ("parallel", "compare", "research multiple")):
        return "parallel_mas"
    if any(k in t for k in ("review", "approve", "compliance", "audit")):
        return "critique_loop"
    if any(k in t for k in ("debug", "incident", "401", "flaky")):
        return "specialist_mas_with_isolation"
    if any(k in t for k in ("release", "deploy", "runbook", "checklist")):
        return "decomposed_or_parallel_mas"
    return "single_call_with_tools"


TASKS = [
    "Translate release notes to French",
    "Parallel research auth and migration docs for release",
    "Review deploy runbook for compliance before prod",
    "Debug incident: flaky 401 on payments endpoint",
    "Prepare production release checklist for Payments API",
]

for task in TASKS:
    print(f"{recommend_architecture(task):<35} ← {task}")

---

## 10. Side-by-Side Comparison on the Same Goal

Run all patterns against `RELEASE_GOAL` and compare measurable outputs.

In [ ]:
async def compare_all_patterns() -> None:
    single_r = single_call_release_checklist(RELEASE_GOAL)
    decomp_r = decomposed_release_review()
    spec_r = specialized_pipeline(RELEASE_GOAL)
    par_r = await parallel_release_review()
    crit_r = critique_loop()

    results = [single_r, decomp_r, spec_r, par_r, crit_r]

    print(f"{'Pattern':<28} {'Calls':>5} {'Latency':>8} {'Citations':>10} {'Quality signal'}")
    print("-" * 80)
    for r in results:
        cites = len(r.get("citations", []))
        if r["pattern"] == "critique_loop":
            quality = f"approved={r.get('approved')}"
        elif cites == 0:
            quality = "no grounding"
        else:
            quality = f"{cites} doc cites"
        print(
            f"{r['pattern']:<28} {r['model_calls']:>5} "
            f"{r['latency_ms']:>7.0f}ms {cites:>10} {quality}"
        )


await compare_all_patterns()

---

## 11. Latency Budget Check

Each agent turn adds **round-trip time** (model + network + tools). Parallel fan-out helps **wall-clock** but not always **billable tokens**.

In [ ]:
LATENCY_BUDGET_MS = 4000  # example SLA

PATTERN_LATENCY = {
    "single_call": 1800,
    "decomposed_sequential": 4200,
    "specialist_sequential": 5100,
    "parallel_research_merge": 2900,
    "critique_loop": 6800,
}

for name, ms in PATTERN_LATENCY.items():
    status = "within SLA" if ms <= LATENCY_BUDGET_MS else "OVER BUDGET"
    bar = "#" * int(ms / 400)
    print(f"{name:<28} {ms:4} ms  [{status:12}] {bar}")

---

## 12. Production Checklist Before Adding Agents

Before shipping MAS to production, verify:

1. Can you name **≥2 specialists** with non-overlapping prompts?
2. Is there a **measurable** quality gain on a golden evaluation set?
3. Do you have **per-agent** traces and token budgets?
4. Is there a **rollback** to single-call if MAS fails?
5. Did stakeholders sign off on **non-deterministic** execution paths?

If quality ties between single-call and MAS on your golden set, **prefer single-call** on cost and latency.

In [ ]:
GOLDEN_SET = [
    {"question": "How to authenticate Payments API?", "expect_doc": "auth-01"},
    {"question": "How to run migrations before deploy?", "expect_doc": "migrate-02"},
    {"question": "What tests must pass?", "expect_doc": "test-03"},
]


def eval_grounding(answer: str, expect_doc: str) -> bool:
    return expect_doc in answer


def run_golden_eval(pattern_fn: Callable[[], dict]) -> float:
    result = pattern_fn()
    answer = result.get("answer", "")
    hits = sum(1 for row in GOLDEN_SET if eval_grounding(answer, row["expect_doc"]))
    return hits / len(GOLDEN_SET)


# Single call: no citations in answer
single_quality = run_golden_eval(lambda: single_call_release_checklist(RELEASE_GOAL))
# Decomposed: has citations
decomp_quality = run_golden_eval(decomposed_release_review)

print(f"Single-call grounding score: {single_quality:.0%}")
print(f"Decomposed grounding score:  {decomp_quality:.0%}")
print("\nIf MAS quality >> single-call, justify the extra cost/latency.")
print("If quality ties, prefer single-call.")

---

## 13. Anti-Pattern — Agents for Appearance

Splitting one prompt into three agents **without** distinct responsibilities adds:

- Coordination bugs (message format drift between agents)
- 3× system prompt overhead
- Harder unit testing and debugging

**Litmus test:** if every agent could share one system message without conflict, you probably do not need MAS.

In [ ]:
def agents_are_justified(agents: list[dict[str, str]]) -> bool:
    """True if agents have meaningfully different system prompts."""
    prompts = [a["system"] for a in agents]
    return len(set(prompts)) == len(prompts) and len(prompts) >= 2


FAKE_MAS = [
    {"name": "agent_1", "system": "You are a helpful assistant."},
    {"name": "agent_2", "system": "You are a helpful assistant."},
    {"name": "agent_3", "system": "You are a helpful assistant."},
]

REAL_MAS = [
    {"name": "retriever", "system": "Return only doc IDs. Never invent."},
    {"name": "critic", "system": "Reject drafts missing citations or auth steps."},
    {"name": "writer", "system": "Merge verified facts into user-facing checklist."},
]

print("Fake MAS justified?", agents_are_justified(FAKE_MAS))
print("Real MAS justified?", agents_are_justified(REAL_MAS))

---

## 14. Optional — Live LLM Comparison

Set `USE_LIVE_LLM = True` with a valid API key to ask a model when MAS beats a single call.

In [ ]:
USE_LIVE_LLM = False

OFFLINE_ANSWER = (
    "MAS beats a single call when you need parallel research across sources, "
    "role-specific prompts that would conflict in one message, critique loops for "
    "quality gates, or isolated retries when one tool step fails."
)

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": "In 2 sentences: when does a multi-agent system beat a single LLM call?",
            }],
            max_tokens=100,
        )
        print(resp.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(OFFLINE_ANSWER)

---

## 15. Check Your Understanding

1. Name three situations where a **single LLM call** is the better choice.
2. Why does **parallel research** reduce wall-clock latency but not always token cost?
3. What problem does a **critique loop** solve, and what cost does it add?
4. How does **failure isolation** in MAS differ from single-call regeneration?
5. In the side-by-side comparison, why does `single_call` score 0% on grounding?
6. What is the **litmus test** for whether agents are justified?
7. If your golden set shows equal quality for single-call and MAS, which do you ship?

---

## 16. Summary

MAS wins on **decomposition**, **specialization**, **parallelism**, **critique**, and **isolation** — not by default. A single `chat.completions.create` call remains the right choice for many tasks.

**Key takeaways:**

- **Start with single-call.** Escalate only when you can name a concrete benefit.
- **Parallel research** is the clearest latency win — independent doc searches run concurrently.
- **Critique loops** improve quality at 2–3× call cost — use for runbooks, compliance, security.
- **Failure isolation** lets you retry one agent without losing good work from others.
- **Measure** cost, latency, and grounding on a golden set before choosing MAS.
- **Anti-pattern:** multiple agents with identical system prompts add overhead without benefit.

Use the decision framework and cost table in design reviews: *Can one call do this? If not, which MAS pattern fits — parallel, specialist, critique, or isolated?*